# Data Engine — download OHLCV candles via CCXT
Fetch and cache OHLCV candles using the `CCXTLoader` connector, which wraps CCXT's unified API.

Cache lives in `data/cache/ccxt/<exchange>/<timeframe>/<symbol>.parquet`.

In [1]:
from quant_research.connectors import CCXTLoader

In [2]:
exchange = "binanceusdm"  # or 'binance', 'bitget', 'okx', ...
timeframe = "1h"
pairs = [
    "BTC/USDT:USDT",
    "ETH/USDT:USDT",
]

loader = CCXTLoader(exchange=exchange)

In [3]:
for pair in pairs:
    loader.download(pair, timeframe, start_date="2024-04-01 00:00:00")

In [4]:
for pair in pairs:
    df = loader.load(pair, timeframe, start_date="2024-04-01 00:00:00", end_date="2024-04-05 00:00:00")
    print(pair)
    print(df)
    print()

BTC/USDT:USDT
shape: (94, 6)
┌─────────────────────┬─────────┬─────────┬─────────┬─────────┬───────────┐
│ datetime            ┆ open    ┆ high    ┆ low     ┆ close   ┆ volume    │
│ ---                 ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---       │
│ datetime[μs]        ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆ f64       │
╞═════════════════════╪═════════╪═════════╪═════════╪═════════╪═══════════╡
│ 2024-04-01 03:00:00 ┆ 70930.1 ┆ 70955.4 ┆ 70588.0 ┆ 70650.1 ┆ 5609.521  │
│ 2024-04-01 04:00:00 ┆ 70650.0 ┆ 70727.2 ┆ 70484.6 ┆ 70512.8 ┆ 5033.489  │
│ 2024-04-01 05:00:00 ┆ 70512.8 ┆ 70595.3 ┆ 69006.5 ┆ 69212.2 ┆ 34363.498 │
│ 2024-04-01 06:00:00 ┆ 69212.2 ┆ 69720.0 ┆ 68895.5 ┆ 69651.7 ┆ 23036.111 │
│ 2024-04-01 07:00:00 ┆ 69651.8 ┆ 69835.0 ┆ 69534.0 ┆ 69637.0 ┆ 8536.026  │
│ …                   ┆ …       ┆ …       ┆ …       ┆ …       ┆ …         │
│ 2024-04-04 20:00:00 ┆ 68483.9 ┆ 68548.8 ┆ 67291.3 ┆ 67945.3 ┆ 36733.783 │
│ 2024-04-04 21:00:00 ┆ 67945.2 ┆ 68167.8 ┆ 67500.0 ┆ 67808

In [5]:
# Market introspection
looking_for = "ETH"
print([s for s in loader.available_symbols if looking_for in s][:20])

['ETH/USDT:USDT', 'ETH/BTC:BTC', 'ETHW/USDT:USDT', 'ETH/USDC:USDC', 'ETHFI/USDT:USDT', 'ETHFI/USDC:USDC', 'NEIROETH/USDT:USDT', 'ETH/USDT:USDT-260626', 'ETH/USDT:USDT-260925']


In [6]:
for symbol in pairs:
    limits = loader.market_limits(symbol)
    ticker = loader.ticker(symbol)
    print(f"{symbol} limits: {limits}")
    print(f"  min notional ~ {limits['amount']['min'] * ticker['close']} quote")
    print()

BTC/USDT:USDT limits: {'leverage': {'min': None, 'max': None}, 'amount': {'min': 0.001, 'max': 1000.0}, 'price': {'min': 556.8, 'max': 4529764.0}, 'cost': {'min': 50.0, 'max': None}, 'market': {'min': 0.001, 'max': 120.0}}
  min notional ~ 78.90060000000001 quote

ETH/USDT:USDT limits: {'leverage': {'min': None, 'max': None}, 'amount': {'min': 0.001, 'max': 10000.0}, 'price': {'min': 39.86, 'max': 306177.0}, 'cost': {'min': 20.0, 'max': None}, 'market': {'min': 0.001, 'max': 2000.0}}
  min notional ~ 2.4172399999999996 quote

